In [7]:
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

chat = ChatOpenAI(model="gpt-4o-mini")
embedding_model = OpenAIEmbeddings()


In [30]:
# Carregando o PDF — cada página vira um Document
caminho = "../files/LLM.pdf"
loader = PyPDFLoader(caminho)
paginas = loader.load()

print(f"Total de páginas carregadas: {len(paginas)}")

recur_split = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

documents = recur_split.split_documents(paginas)
print(f"Total de chunks gerados: {len(documents)}")
print(f"\nExemplo do primeiro chunk:")
print(documents[0].page_content)

Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 155 0 (offset 0)
Ignoring wrong pointing object 158 0 (offset 0)
Ignoring wrong pointing object 160 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignori

Total de páginas carregadas: 9
Total de chunks gerados: 47

Exemplo do primeiro chunk:
E-BOOK 
Um guia compacto sobre Large Language Models (LLM)


In [31]:
diretorio = 'files/chat_retrieval_db'

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=diretorio
)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,        # quantos retornar no final
        "fetch_k": 10  # quantos candidatos buscar antes de filtrar
    }
)

In [32]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
        "Você é um assistente especialista. "
        "Responda usando apenas o contexto abaixo:\n\n{context}"),
    ("human", "{input}"),
])

In [33]:
# Monta a chain manualmente com LCEL (LangChain Expression Language)
chat_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | chat
    | StrOutputParser()
)

In [34]:
resposta = chat_chain.invoke("O que é uma lista em Python?")
print(resposta)

Uma lista em Python é um conjunto sequencial de valores, onde cada valor é identificado através de um índice. O primeiro valor tem índice 0. Uma lista pode conter valores de qualquer tipo, incluindo outras listas. É declarada da seguinte forma: 

```python
Nome_Lista = [ valor1, valor2, ..., valorN]
```

Por exemplo:

```python
L = [3, 'abacate', 9.7, [5, 6, 3], "Python", (3, 'j')]
```

Os elementos da lista podem ser acessados através de seus índices.


In [35]:
from langchain_core.globals import set_debug

set_debug(True)
resposta = chat_chain.invoke("O que é Hugging Face e como faço para acessa-lo?")
print(resposta)
set_debug(False)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "O que é Hugging Face e como faço para acessa-lo?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,input>] Entering Chain run with input:
{
  "input": "O que é Hugging Face e como faço para acessa-lo?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,input> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "O que é Hugging Face e como faço para acessa-lo?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,input> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "O que é Hugging Face e como faço para acessa-lo?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,input>] s] Exiting Chain run with output:
[outputs]
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [chain:RunnableSequence > prompt:Chat